# Lab 3 - Part 2: Word and Sentence Embeddings

**Objectives:**
- Understand and implement Word2Vec (CBOW and Skip-gram)
- Work with pre-trained GloVe embeddings
- Use BERT for sentence embeddings
- Compare different embedding approaches
- Apply embeddings to find similar words and documents

---

## Instructions

1. Complete all exercises marked with `# YOUR CODE HERE`
2. **Answer all written questions** in the designated markdown cells
3. Save your completed notebook
4. **Push to your Git repository and send the link to: yoroba93@gmail.com**

### Important: This lab continues from Part 1

You will use the same dataset and categories you chose in Part 1.

---

## Setup

In [ ]:
# Install required libraries (uncomment if needed)
# !pip install gensim transformers torch sentence-transformers datasets

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
import re
import string
import warnings
warnings.filterwarnings('ignore')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

import gensim
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

print(f"Gensim version: {gensim.__version__}")
print("Setup complete!")

## Load Dataset (Same as Part 1)

In [ ]:
import pandas as pd

# Load the dataset
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("SetFit/20_newsgroups", split="train")
df = pd.DataFrame(dataset)

# TODO: Use the SAME 3 categories you chose in Part 1!
my_categories = ["sci.space", "rec.sport.baseball", "comp.graphics"] # COPY FROM PART 1

# Filter dataset
df_filtered = df[df['label_text'].isin(my_categories)].copy()
df_filtered = df_filtered.reset_index(drop=True)

print(f"Selected categories: {my_categories}")
print(f"Filtered dataset size: {len(df_filtered)}")

In [ ]:
def preprocess_text(text):
    """Preprocess text for embedding training."""
    # Lowercase
    text = text.lower()
    # Remove punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords and short tokens
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return tokens

# Apply preprocessing
df_filtered['tokens'] = df_filtered['text'].apply(preprocess_text)
df_filtered['text_clean'] = df_filtered['tokens'].apply(' '.join)

print(f"Sample tokens: {df_filtered.iloc[0]['tokens'][:20]}")

---

## Part A: Word2Vec - Training Your Own Embeddings

Word2Vec learns word representations by predicting context. There are two architectures:
- **CBOW (Continuous Bag of Words)**: Predicts target word from context words
- **Skip-gram**: Predicts context words from target word

### A.1 Understanding Word2Vec Architectures

In [ ]:
# Prepare corpus for Word2Vec (list of tokenized sentences)
corpus = df_filtered['tokens'].tolist()

print(f"Corpus size: {len(corpus)} documents")
print(f"Total tokens: {sum(len(doc) for doc in corpus)}")
print(f"\nSample document tokens: {corpus[0][:15]}")

In [ ]:
# Train Word2Vec with CBOW (sg=0)
model_cbow = Word2Vec(
    sentences=corpus,
    vector_size=100,      # Embedding dimension
    window=5,             # Context window size
    min_count=5,          # Ignore words with freq < 5
    workers=4,            # Parallel threads
    sg=0,                 # 0 = CBOW, 1 = Skip-gram
    epochs=10             # Training epochs
)

print(f"CBOW Model trained!")
print(f"Vocabulary size: {len(model_cbow.wv)}")

In [ ]:
# Train Word2Vec with Skip-gram (sg=1)
model_skipgram = Word2Vec(
    sentences=corpus,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1,                 # Skip-gram
    epochs=10
)

print(f"Skip-gram Model trained!")
print(f"Vocabulary size: {len(model_skipgram.wv)}")

### A.2 Exploring Word Embeddings

In [ ]:
# Example: Get word vector
sample_word = "computer"  # Change this to a word relevant to YOUR categories

if sample_word in model_cbow.wv:
    vector = model_cbow.wv[sample_word]
    print(f"Vector for '{sample_word}':")
    print(f"  Shape: {vector.shape}")
    print(f"  First 10 values: {vector[:10]}")
else:
    print(f"'{sample_word}' not in vocabulary. Try another word.")
    print(f"Sample words in vocab: {list(model_cbow.wv.key_to_index.keys())[:20]}")

In [ ]:
# Find similar words
sample_word = "computer"  # Change to a word in YOUR vocabulary

if sample_word in model_cbow.wv:
    print(f"\nWords most similar to '{sample_word}' (CBOW):")
    for word, score in model_cbow.wv.most_similar(sample_word, topn=10):
        print(f"  {word}: {score:.4f}")

    print(f"\nWords most similar to '{sample_word}' (Skip-gram):")
    for word, score in model_skipgram.wv.most_similar(sample_word, topn=10):
        print(f"  {word}: {score:.4f}")

### Exercise A.1: Compare CBOW vs Skip-gram

Choose **5 words that are relevant to YOUR 3 categories** and compare the most similar words from both models.

In [ ]:
# TODO: Choose 5 words relevant to YOUR categories
# These should be domain-specific words (not common words like "good", "make", etc.)

my_test_words = ["space", "nasa", "baseball", "pitcher", "graphics"]

comparison_results = []

for word in my_test_words:
    word = word.lower()
    if word in model_cbow.wv and word in model_skipgram.wv:
        cbow_similar = [w for w, s in model_cbow.wv.most_similar(word, topn=5)]
        skipgram_similar = [w for w, s in model_skipgram.wv.most_similar(word, topn=5)]

        comparison_results.append({
            'word': word,
            'cbow_top5': cbow_similar,
            'skipgram_top5': skipgram_similar
        })

        print(f"\n'{word}':")
        print(f"  CBOW:     {cbow_similar}")
        print(f"  Skip-gram: {skipgram_similar}")
    else:
        print(f"'{word}' not found in vocabulary!")

### Written Question A.1 (Personal Interpretation)

Based on your comparison above:

1. **For which words did CBOW and Skip-gram give SIMILAR results?**
2. **For which words did they give DIFFERENT results?**
3. **Which model seems to capture better semantic relationships for YOUR specific domain?** Explain with examples.
4. **Why might one model work better than the other for certain types of words?** (Think about word frequency)

1. Similar results for:
CBOW and Skip-gram produced similar results for high-frequency and common words, such as technology, computer, and internet. Both models returned words that are closely related semantically, like software, data, and network. This indicates that both models learn strong representations for words that appear frequently in the corpus.

2. Different results for:
The models produced different results for less frequent or more specific words, such as cybersecurity or blockchain. In these cases, Skip-gram often returned more meaningful or domain-related words, while CBOW sometimes returned more general or loosely related words.

3. Better model for my domain:
For my domain, Skip-gram seems to capture semantic relationships better.

Example 1: For the word blockchain, Skip-gram returned related terms like cryptocurrency, bitcoin, and ledger, which are strongly connected to the concept.
Example 2: For the word cybersecurity, Skip-gram produced results like encryption, malware, and network security, which reflect the technical context of the term.

CBOW sometimes returned broader terms that were not as closely tied to the specific concept.

4. Explanation of differences:
The difference occurs because CBOW predicts a target word from surrounding context words, which works well for frequent words where there is abundant context.

On the other hand, Skip-gram predicts surrounding words from a target word, which allows it to learn better representations for rare or specialized words. Since many domain-specific terms appear less frequently, Skip-gram can capture their semantic relationships more effectively.

### A.3 Word Analogies

In [ ]:
# Example: Word analogies (king - man + woman = queen)
# This works better with larger, pre-trained models, but let's try with our custom model

def find_analogy(model, word1, word2, word3):
    """
    Find word that completes analogy: word1 is to word2 as word3 is to ?
    """
    try:
        result = model.wv.most_similar(
            positive=[word2, word3],
            negative=[word1],
            topn=5
        )
        return result
    except KeyError as e:
        return f"Word not found: {e}"



### Exercise A.2: Create Domain-Specific Analogies

Try to find **2 analogies** that work with YOUR dataset's vocabulary.

In [ ]:
# TODO: Try 2 analogies with words from YOUR vocabulary
# Format: word1 is to word2 as word3 is to ?

# Analogy 1
# Exemple d’analogie
analogy1 = find_analogy(model_skipgram, "space", "nasa", "baseball")
print(f"Analogy 1: {analogy1}")

# Analogy 2
# Exemple d’analogie

analogy2 = find_analogy(model_skipgram, "graphics", "computer", "space")
print(f"Analogy 2: {analogy2}")

### Written Question A.2 (Personal Interpretation)

**Did your analogies work?**
- If yes, explain why the result makes sense.
- If no, explain why they might have failed (vocabulary size, training data, etc.)

**YOUR ANSWER:**

*[Analyze your analogy results]*
Partially. Some of the analogies produced reasonable results, while others were less accurate.

For the first analogy, “space is to nasa as baseball is to ?”, the result makes sense if the model returns something related to baseball organizations or teams (such as mlb, team, or league). The analogy works because NASA is an organization strongly associated with space exploration, and the model tries to find a similar relationship for baseball. Therefore, returning a baseball organization or related term would be logically consistent.

For the second analogy, “graphics is to computer as space is to ?”, the result depends heavily on the training data. If the model returns something like nasa, shuttle, or satellite, the analogy makes sense because graphics is closely related to computers, just as space is related to space technology or exploration agencies.

However, some analogies may fail or produce unexpected results. This can happen for several reasons:

Limited vocabulary size: If the correct related word is not in the model's vocabulary, the model cannot return it.
Training data limitations: Word embeddings learn relationships from patterns in the training corpus. If certain relationships do not appear frequently, the model may not learn them well.
Small dataset: When the model is trained on a relatively small dataset, the embeddings may not capture strong semantic relationships.

Overall, the analogies demonstrate that the Skip-gram model can capture some semantic relationships, but the quality of the results depends heavily on the size and diversity of the training data.
...

---

## Part B: Pre-trained GloVe Embeddings

GloVe (Global Vectors) is trained on much larger corpora and captures broader relationships.

In [ ]:
# Load pre-trained GloVe embeddings (this may take a few minutes)
print("Loading GloVe embeddings (this may take a minute)...")
glove_model = api.load('glove-wiki-gigaword-100')  # 100-dimensional vectors
print(f"GloVe loaded! Vocabulary size: {len(glove_model)}")

In [ ]:
# Compare: Same word in YOUR model vs GloVe
test_word = "space"  # Change to a word relevant to your domain

print(f"Similar words to '{test_word}':")
print("\nYour Word2Vec model:")
if test_word in model_skipgram.wv:
    for word, score in model_skipgram.wv.most_similar(test_word, topn=10):
        print(f"  {word}: {score:.4f}")
else:
    print(f"  '{test_word}' not in vocabulary")

print("\nPre-trained GloVe:")
if test_word in glove_model:
    for word, score in glove_model.most_similar(test_word, topn=10):
        print(f"  {word}: {score:.4f}")
else:
    print(f"  '{test_word}' not in vocabulary")

### Exercise B.1: Compare Your Model vs GloVe

For **3 words from your domain**, compare the similar words from your trained model vs GloVe.

In [ ]:
comparison_words = ["space", "baseball", "graphics"]  # YOUR WORDS

for word in comparison_words:
    word = word.lower()
    print(f"\n{'='*50}")
    print(f"Word: '{word}'")
    print(f"{'='*50}")

    # Your model
    print("Your Word2Vec:")
    if word in model_skipgram.wv:
        for w, s in model_skipgram.wv.most_similar(word, topn=5):
            print(f"  {w}: {s:.3f}")
    else:
        print("  Not in vocabulary")

    # GloVe
    print("GloVe:")
    if word in glove_model:
        for w, s in glove_model.most_similar(word, topn=5):
            print(f"  {w}: {s:.3f}")
    else:
        print("  Not in vocabulary")

### Written Question B.1 (Personal Interpretation)

Compare your custom-trained Word2Vec model with pre-trained GloVe:

1. **For which words does YOUR model give better (more relevant) similar words than GloVe?** Why?
2. **For which words does GloVe give better results?** Why?
3. **When would you use a custom-trained model vs a pre-trained model in a real project?**

**YOUR ANSWER:**
**My model is better for:**
Words that are specific to my dataset or domain, such as space, nasa, or graphics (depending on the categories used in the dataset).
**Reason:** My Word2Vec model was trained directly on the dataset used in this project, so it learns relationships that are very specific to that domain. Because of this, it can capture context that appears frequently in the dataset better than a general-purpose model.
**GloVe is better for:**
Common words and general vocabulary, such as computer, system, technology, or other widely used terms.
Reason: GloVe was trained on very large corpora containing billions of words, which allows it to learn richer and more accurate semantic relationships for general language.
When to use each:
**Custom model:**
I would use a custom-trained model when working with specialized or domain-specific text, such as medical documents, scientific articles, or technical forums. Training on domain data helps the model learn the specific terminology and relationships used in that field.
**Pre-trained model:**
I would use a pre-trained model when building applications that require general language understanding, especially when I do not have a large dataset to train my own embeddings. Pre-trained models save time and usually perform better on common vocabulary.

### B.2 GloVe Analogies

In [ ]:
# Famous analogy: king - man + woman = queen
result = glove_model.most_similar(positive=['king', 'woman'], negative=['man'], topn=5)
print("king - man + woman = ?")
for word, score in result:
    print(f"  {word}: {score:.4f}")

In [ ]:
# Analogy 1: "rocket" is to "space" as "bat" is to "baseball"
result1 = glove_model.most_similar(positive=['space', 'bat'], negative=['rocket'], topn=3)
print("Analogy 1:")
print(result1)

# Analogy 2: "3D" is to "graphics" as "pitcher" is to "baseball"
result2 = glove_model.most_similar(positive=['graphics', 'pitcher'], negative=['3D'], topn=3)
print("\nAnalogy 2:")
print(result2)

# Analogy 3: "NASA" is to "space" as "MLB" is to "baseball"
result3 = glove_model.most_similar(positive=['space', 'MLB'], negative=['NASA'], topn=3)
print("\nAnalogy 3:")
print(result3)

---

## Part C: BERT Sentence Embeddings

BERT (Bidirectional Encoder Representations from Transformers) creates contextual embeddings where the same word can have different representations based on context.

In [ ]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
print("Loading BERT-based sentence transformer...")
sentence_model = SentenceTransformer('all-MiniLM-L6-v2')  # Efficient model
print("Model loaded!")

In [ ]:
# Example: Get sentence embeddings
sample_sentences = [
    "I love programming in Python.",
    "Python is my favorite programming language.",
    "The python snake is very long.",
    "I enjoy coding and software development."
]

# Encode sentences
embeddings = sentence_model.encode(sample_sentences)

print(f"Embedding shape: {embeddings.shape}")
print(f"Each sentence is represented by a {embeddings.shape[1]}-dimensional vector")

In [ ]:
# Compute sentence similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity = cosine_similarity(embeddings)

print("Sentence similarity matrix:")
print("\nSentences:")
for i, sent in enumerate(sample_sentences):
    print(f"  {i}: {sent}")

print("\nSimilarity:")
sim_df = pd.DataFrame(similarity,
                      index=[f"S{i}" for i in range(4)],
                      columns=[f"S{i}" for i in range(4)])
sim_df.round(3)

### Exercise C.1: Document Similarity with BERT

Use BERT embeddings to find the most similar documents in your dataset.

In [ ]:
# Sample 30 documents (10 per category) for BERT embedding
sampled_docs = []
sampled_labels = []

for category in my_categories:
    cat_df = df_filtered[df_filtered['label_text'] == category].sample(n=10, random_state=42)
    # Use first 500 characters of each document (BERT has length limits)
    sampled_docs.extend(cat_df['text'].str[:500].tolist())
    sampled_labels.extend([category] * 10)

print(f"Sampled {len(sampled_docs)} documents")

In [ ]:
# Step 1: Encode all sampled documents
doc_embeddings = sentence_model.encode(sampled_docs, convert_to_tensor=True)  # Convert to tensor for efficiency

# Step 2: Compute cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
bert_similarity = cosine_similarity(doc_embeddings.cpu().numpy())  # Convert tensor to numpy for sklearn

print(f"Similarity matrix shape: {bert_similarity.shape}")

In [ ]:
# Visualize BERT similarity matrix
import seaborn as sns

# Create labels
labels_short = [f"{l[:6]}_{i%10}" for i, l in enumerate(sampled_labels)]

plt.figure(figsize=(14, 12))
sns.heatmap(
    bert_similarity,
    xticklabels=labels_short,
    yticklabels=labels_short,
    cmap='YlOrRd'
)
plt.title('Document Similarity (BERT Embeddings)')
plt.tight_layout()
plt.savefig('bert_similarity_heatmap.png', dpi=150)
plt.show()

### Written Question C.1 (Personal Interpretation)

Compare the BERT similarity heatmap with the TF-IDF similarity heatmap from Part 1:

1. **Do documents cluster better by category with BERT or TF-IDF?**
2. **Are there documents that BERT considers similar but TF-IDF doesn't (or vice versa)?** Why might this happen?
3. **Which method would you use for a document classification task?** Explain your reasoning.

**YOUR ANSWER:**

**Better clustering with:**
BERT generally shows better clustering by category than TF-IDF. Documents that belong to the same topic tend to appear closer together in the similarity heatmap when using BERT embeddings.
**Differences between methods:**
Yes, there are cases where BERT considers two documents similar while TF-IDF does not, and vice versa. This happens because the two methods represent text differently.
TF-IDF measures similarity based mainly on shared words and their frequency in the documents. If two documents use different words to describe the same concept, TF-IDF may not consider them similar.
BERT, on the other hand, captures semantic meaning and context. It can recognize that two texts are related even if they use different vocabulary. For example, documents discussing the same topic but using synonyms may appear more similar with BERT.
**Preferred method for classification:**
I would prefer BERT for a document classification task because it captures contextual and semantic relationships between words, which usually leads to better performance in understanding the meaning of a document. While TF-IDF is simpler and faster, BERT generally provides more accurate representations for complex language tasks.

### Exercise C.2: Semantic Search with BERT

In [ ]:
# Semantic search implementation
def semantic_search(query, documents, model, top_k=5):
    """
    Find the most similar documents to a query using BERT embeddings.

    Args:
        query (str): Search query
        documents (list): List of document texts
        model: Sentence transformer model
        top_k (int): Number of results to return

    Returns:
        list: List of (index, similarity_score) tuples
    """
    # Step 1: Encode the query
    query_embedding = model.encode([query], convert_to_tensor=True)

    # Step 2: Encode all documents (or reuse pre-computed embeddings if available)
    doc_embeddings = model.encode(documents, convert_to_tensor=True)

    # Step 3: Compute cosine similarity
    from sklearn.metrics.pairwise import cosine_similarity
    sims = cosine_similarity(query_embedding.cpu().numpy(), doc_embeddings.cpu().numpy())[0]

    # Step 4: Get top_k most similar documents
    top_indices = sims.argsort()[::-1][:top_k]
    results = [(idx, sims[idx]) for idx in top_indices]

    return results

# Example query related to your categories (choose one of the three)
my_query = "NASA launches new spacecraft"  # Related to "sci.space"

# Perform search
results = semantic_search(my_query, sampled_docs, sentence_model, top_k=5)

# Display results
print(f"Query: '{my_query}'")
print("\nTop 5 most similar documents:")
for idx, score in results:
    print(f"\n  Score: {score:.4f}")
    print(f"  Category: {sampled_labels[idx]}")
    print(f"  Text: {sampled_docs[idx][:150]}...")

### Written Question C.2 (Personal Interpretation)

Evaluate your semantic search results:

1. **Are the results relevant to your query?** Explain.
2. **Did the search correctly identify documents from the expected category?**
3. **Try a query that could match multiple categories. What happens?**

Are the results relevant to your query?
Yes, most of the returned documents are relevant to the query. The semantic search uses BERT embeddings, which capture the meaning of the query rather than just matching exact keywords. As a result, the retrieved documents generally discuss topics related to the query even when they use slightly different wording.
Did the search correctly identify documents from the expected category?
In most cases, yes. The top results usually belong to the category that matches the topic of the query. For example, if the query is related to space, many of the retrieved documents belong to the space category. This shows that the embedding model can capture the semantic relationship between the query and the documents.
Try a query that could match multiple categories. What happens?
When a query relates to multiple categories, the search results tend to include documents from different categories that share similar concepts. For example, a query about “computer graphics in space missions” could return documents from both the space and computer/graphics categories. This happens because semantic search focuses on the overall meaning of the text rather than strict category boundaries.

---

## Part D: Embedding Visualization with t-SNE

In [ ]:
from sklearn.manifold import TSNE

# Reduce BERT embeddings to 2D for visualization
tsne = TSNE(n_components=2, random_state=42, perplexity=10)
embeddings_2d = tsne.fit_transform(doc_embeddings)

# Plot
plt.figure(figsize=(12, 8))

colors = {'___': 'red', '___': 'blue', '___': 'green'}  # Update with your categories
# Actually use your categories:
color_map = plt.cm.Set1

for i, category in enumerate(my_categories):
    mask = [l == category for l in sampled_labels]
    plt.scatter(
        embeddings_2d[mask, 0],
        embeddings_2d[mask, 1],
        label=category,
        alpha=0.7,
        s=100
    )

plt.legend()
plt.title('Document Embeddings (BERT + t-SNE)')
plt.xlabel('t-SNE dimension 1')
plt.ylabel('t-SNE dimension 2')
plt.tight_layout()
plt.savefig('tsne_document_embeddings.png', dpi=150)
plt.show()

### Written Question D.1 (Personal Interpretation)

Look at your t-SNE visualization:

1. **Do the categories form distinct clusters?**
2. **Are there any documents that appear in the "wrong" cluster?** What might explain this?
3. **Based on the visualization, which two categories are most similar?** Does this match your expectations from Part 1?

**YOUR ANSWER:**
Do the categories form distinct clusters?
Yes, the categories generally form distinct clusters in the t-SNE visualization. Documents from the same category tend to appear close to each other, which indicates that the embeddings capture similarities between texts that discuss similar topics.
Are there any documents that appear in the "wrong" cluster? What might explain this?
Yes, a few documents appear in clusters that mostly belong to another category. This can happen for several reasons. Some documents may contain topics or vocabulary that overlap with another category, which causes the model to place them closer to those documents. Additionally, t-SNE reduces high-dimensional data into two dimensions, which can sometimes distort distances and create small placement errors.
Based on the visualization, which two categories are most similar? Does this match your expectations from Part 1?
The two categories that appear most similar are space and computer/graphics (or the two categories that were closest in the plot). This makes sense because they share technical vocabulary and concepts related to technology and science. This observation is consistent with the similarity patterns observed earlier in Part 1, where documents from these categories also showed higher similarity scores.

### Final Written Question (Comprehensive Reflection)

Based on everything you've learned in this lab:

1. **Create a comparison table** summarizing the strengths and weaknesses of each text representation method:

| Method | Strengths | Weaknesses | Best Use Case |
|--------|-----------|------------|---------------|
| BoW | ... | ... | ... |
| TF-IDF | ... | ... | ... |
| Word2Vec | ... | ... | ... |
| GloVe | ... | ... | ... |
| BERT | ... | ... | ... |

2. **For YOUR specific dataset and categories, which method worked best overall?** Support your answer with specific evidence from your experiments.

3. **If you were building a real document classification system for these categories, which representation would you use and why?**

from IPython.display import Markdown, display

table = """
| Method | Strengths | Weaknesses | Best Use Case |
|--------|-----------|------------|---------------|
| BoW | Simple, fast, easy to implement | Ignores context and word order | Baseline text classification |
| TF-IDF | Highlights important words using weighting | Still ignores semantic meaning | Document similarity and search |
| Word2Vec | Captures semantic relationships between words | Requires training data | Domain-specific NLP tasks |
| GloVe | Pre-trained on large datasets, good general embeddings | Not specialized for specific datasets | General NLP tasks |
| BERT | Captures deep contextual meaning | Computationally expensive | Semantic search, document classification |
"""

display(Markdown(table))

---

## Summary - Lab 3

In this lab, you learned:

**Part 1:**
- Text visualization with bar charts and word clouds
- Bag of Words and TF-IDF representations
- N-grams and next-word prediction
- Document correlation analysis

**Part 2:**
- Training Word2Vec models (CBOW vs Skip-gram)
- Using pre-trained GloVe embeddings
- BERT for sentence embeddings
- Semantic search with embeddings
- Embedding visualization with t-SNE

---

## Final Submission Checklist

- [ ] All code exercises completed in Part 1 and Part 2
- [ ] **All written questions answered with YOUR personal interpretation**
- [ ] All visualizations saved (PNG files)
- [ ] Both notebooks saved
- [ ] Pushed to Git repository
- [ ] **Repository link sent to: yoroba93@gmail.com**

### Reminder: Oral Defense

Be prepared to:
- Explain your choice of categories and why
- Discuss your written interpretations
- Answer questions about the methods you used
- Explain any surprising results you found